In [ ]:
import pandas as pd
from time import sleep
from tqdm import tqdm
from normdata.utils import import_from_normdata
from csae_pyutils import gsheet_to_df
from apis_core.apis_metainfo.models import Collection, Uri
from apis_core.apis_entities.models import Institution


In [ ]:
col, _ = Collection.objects.get_or_create(name="DLA-Schnitzler")
domain = "dla-marbach"
col.save()

In [ ]:
df = gsheet_to_df("1ah4KXPeihzk0Rn10IcRxI-DvobsCoy8evuhSx-e-LNE")

In [ ]:
df.head()

In [ ]:
broken_gnd = []
errors = []
already_exist = []
for i, row in tqdm(df.iterrows()):
    tmp_uri = row["dla_url"]
    if pd.isna(row["gnd"]) and pd.isna(row["wikidata"]):
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
        entity = pmb_uri.entity
        if entity is None:
            entity = Institution.objects.create(name=row["name"])
            pmb_uri.entity = entity
            pmb_uri.save()
            entity.collection.add(col)
            continue
        else:
            continue
    elif pd.isna(row["gnd"]) and row["wikidata"]:
        norm_uri = f'http://www.wikidata.org/entity/{row["wikidata"]}'
    else:
        gnd = row["gnd"]
        norm_uri = f"https://d-nb.info/gnd/{gnd}"
    try:
        entity = import_from_normdata(norm_uri, "institution")
    except Exception as e:
        errors.append([norm_uri, e])
    sleep(0.5)
    if entity:
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
        pmb_uri.entity = entity
        pmb_uri.save()
        entity.collection.add(col)
    else:
        broken_gnd.append([tmp_uri, norm_uri])

In [ ]:
errors